# 🚀 Carnet 04 : Transformers Arabes Pré-entraînés (PLMs)
## Feature Extraction (Gelé), Fine-Tuning SOTA (AraBERT vs MARBERTv2) & Apprentissage Multi-Tâche

**Projet M2 : Détection de Position (*Stance Detection*) en Arabe sur Twitter (Dataset Mawqif)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Objectifs de ce Carnet (Parties D et E) :
1. **Partie D - Modèle Gelé (*Feature Extraction*) avec `AraBERTv02-Twitter` :**
   * **D1 :** Extraction des représentations via le token `[CLS]` vs la moyenne des tokens (*Mean Pooling*) suivie d'une régression logistique (`class_weight='balanced'`).
   * **D2 :** Branchement des états de sortie de BERT sur notre tête séquentielle **BiLSTM + Attention** (de C2) avec **visualisation des poids d'attention sur 5 exemples concrets**.
2. **Partie E1 - Fine-Tuning SOTA : AraBERT-Twitter vs MARBERTv2 :**
   * Entrée formatée en paire : `[CLS] Cible [SEP] Tweet [SEP]`.
   * Entraînement complet des 110M de paramètres avec taux d'apprentissage de $2\times 10^{-5}$, warmup de 10%, perte pondérée et arrêt précoce.
   * Évaluation rigoureuse sur **3 graines aléatoires (seeds : 42, 123, 999)** pour mesurer la stabilité statistique (réponse à **Q8**).
3. **Partie E2 - Apprentissage Multi-Tâche (*Multi-Task Learning*) :**
   * Architecture à 3 têtes : **Position (Stance)** + **Sentiment** + **Sarcasme**.
   * Optimisation conjointe : $\mathcal{L}_{\text{totale}} = \mathcal{L}_{\text{stance}} + \lambda_1 \mathcal{L}_{\text{sentiment}} + \lambda_2 \mathcal{L}_{\text{sarcasme}}$.
   * Étude de l'impact des tâches auxiliaires et du paramètre $\lambda$ (réponse à **Q9**).


---
## 1. Environnement, Installation & Détection GPU


In [1]:
# Installation et clonage automatique pour Google Colab
import sys, os

if 'google.colab' in str(get_ipython()):
    print("Google Colab détecté. Préparation des dépendances...")
    !pip install -q transformers torch accelerate scikit-learn matplotlib seaborn
    !mkdir -p data results src
    # Téléchargement direct des fichiers si besoin
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Train.csv" -o data/Mawqif_AllTargets_Train.csv
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Test.csv" -o data/Mawqif_AllTargets_Test.csv

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup

# Détection de l'accélérateur GPU (T4 / V100 / A100)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")
if torch.cuda.is_available():
    print(f"Modèle GPU : {torch.cuda.get_device_name(0)}")


Google Colab détecté. Préparation des dépendances...
Device utilisé : cuda
Modèle GPU : Tesla T4


---
## 2. Chargement des Données & Alignement Multi-Tâches
Préparation du split stratifié à 10% avec encodage des 3 étiquettes : **Position**, **Sentiment**, et **Sarcasme**.


In [2]:
def compute_stance_metrics(y_true, y_pred, labels=["Favor", "Against", "None"]):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    f1_dict = {f"F1_{lbl}": round(float(val), 4) for lbl, val in zip(labels, f1)}
    f1_dict["F1_StanceEval"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"]) / 2.0, 4)
    f1_dict["Macro_F1"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"] + f1_dict["F1_None"]) / 3.0, 4)
    return f1_dict

train_path = "data/Mawqif_AllTargets_Train.csv" if os.path.exists("data/Mawqif_AllTargets_Train.csv") else "../data/Mawqif_AllTargets_Train.csv"
test_path = "data/Mawqif_AllTargets_Test.csv" if os.path.exists("data/Mawqif_AllTargets_Test.csv") else "../data/Mawqif_AllTargets_Test.csv"

# Chargement avec correction de l'artefact CSV ('None' non transformé en NaN)
train_df = pd.read_csv(train_path, keep_default_na=False, encoding="utf-8-sig")
test_df = pd.read_csv(test_path, keep_default_na=False, encoding="utf-8-sig")

# Stratification 10% (Target x Stance)
train_df['strat_key'] = train_df['target'].astype(str) + "___" + train_df['stance'].astype(str)
train_sub, val_df = train_test_split(train_df, test_size=0.10, stratify=train_df['strat_key'], random_state=42)
train_sub = train_sub.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Mappages numériques pour les 3 tâches
stance2id = {"Against": 0, "Favor": 1, "None": 2}
id2stance = {0: "Against", 1: "Favor", 2: "None"}

sentiment2id = {"Negative": 0, "Positive": 1, "Neutral": 2}
sarcasm2id = {"No": 0, "Yes": 1}

# Application des mappages
for df in [train_sub, val_df, test_df]:
    df['stance_id'] = df['stance'].map(stance2id)
    df['sentiment_id'] = df['sentiment'].map(sentiment2id).fillna(2).astype(int)
    df['sarcasm_id'] = df['sarcasm'].map(sarcasm2id).fillna(0).astype(int)

y_val_stance = val_df['stance'].values
print(f"Jeu Train : {len(train_sub)} | Validation : {len(val_df)} | Test : {len(test_df)}")


Jeu Train : 3151 | Validation : 351 | Test : 619


---
## 3. Partie D1 : AraBERTv02-Twitter Gelé (*Feature Extraction*)
Comparaison de deux stratégies d'extraction :
1. **`[CLS]` Pooling** : l'état du tout premier token.
2. **Mean Pooling** : la moyenne de tous les états cachés des tokens réels.
Suivies d'une **Régression Logistique** avec pondération des classes (`class_weight='balanced'`).


In [3]:
arabert_name = "aubmindlab/bert-base-arabertv02-twitter"
print(f"Chargement du modèle et tokenizer : {arabert_name}...")
tokenizer_ara = AutoTokenizer.from_pretrained(arabert_name)
model_ara_frozen = AutoModel.from_pretrained(arabert_name).to(device)
model_ara_frozen.eval()

# Gel strict des poids
for p in model_ara_frozen.parameters():
    p.requires_grad = False

def extract_bert_features(df, tokenizer, model, batch_size=32):
    cls_features = []
    mean_features = []

    # Entrée en paire : Cible [SEP] Tweet
    targets = df['target'].tolist()
    texts = df['text'].tolist()

    for i in tqdm(range(0, len(df), batch_size), desc="Extraction BERT"):
        b_targets = targets[i:i+batch_size]
        b_texts = texts[i:i+batch_size]

        encoded = tokenizer(
            b_targets,
            b_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
            hidden_states = outputs.last_hidden_state # (B, Seq_len, 768)

            # 1. CLS Pooling : token à l'indice 0
            cls_rep = hidden_states[:, 0, :].cpu().numpy()
            cls_features.append(cls_rep)

            # 2. Mean Pooling : moyenne pondérée par le masque d'attention
            mask = encoded['attention_mask'].unsqueeze(-1).float() # (B, Seq_len, 1)
            sum_hidden = (hidden_states * mask).sum(dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            mean_rep = (sum_hidden / sum_mask).cpu().numpy()
            mean_features.append(mean_rep)

    return np.vstack(cls_features), np.vstack(mean_features)

print("Extraction des features sur Train sub...")
X_train_cls, X_train_mean = extract_bert_features(train_sub, tokenizer_ara, model_ara_frozen)
print("Extraction des features sur Validation...")
X_val_cls, X_val_mean = extract_bert_features(val_df, tokenizer_ara, model_ara_frozen)

y_train_id = train_sub['stance_id'].values
y_val_id = val_df['stance_id'].values

# Évaluation Régression Logistique sur [CLS]
clf_cls = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf_cls.fit(X_train_cls, y_train_id)
preds_cls = [id2stance[i] for i in clf_cls.predict(X_val_cls)]
m_d1_cls = compute_stance_metrics(y_val_stance, preds_cls)
print("-> D1 : AraBERT Gelé [CLS] + LogReg :", m_d1_cls)

# Évaluation Régression Logistique sur Mean Pooling
clf_mean = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf_mean.fit(X_train_mean, y_train_id)
preds_mean = [id2stance[i] for i in clf_mean.predict(X_val_mean)]
m_d1_mean = compute_stance_metrics(y_val_stance, preds_mean)
print("-> D1 : AraBERT Gelé [Mean Pooling] + LogReg :", m_d1_mean)


Chargement du modèle et tokenizer : aubmindlab/bert-base-arabertv02-twitter...


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Extraction des features sur Train sub...


Extraction BERT:   0%|          | 0/99 [00:00<?, ?it/s]

Extraction des features sur Validation...


Extraction BERT:   0%|          | 0/11 [00:00<?, ?it/s]

-> D1 : AraBERT Gelé [CLS] + LogReg : {'F1_Favor': 0.7805, 'F1_Against': 0.6108, 'F1_None': 0.2697, 'F1_StanceEval': 0.6956, 'Macro_F1': 0.5537}
-> D1 : AraBERT Gelé [Mean Pooling] + LogReg : {'F1_Favor': 0.7861, 'F1_Against': 0.6415, 'F1_None': 0.4091, 'F1_StanceEval': 0.7138, 'Macro_F1': 0.6122}


---
## 4. Partie D2 : AraBERT Gelé + Tête BiLSTM + Attention
On connecte les représentations de chaque token de BERT gelé à une tête **BiLSTM + Attention** (de C2) et on visualise les poids d'attention sur 5 exemples concrets.


In [4]:
import torch.nn.functional as F
import torch.nn as nn
class Attention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=64):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, rnn_outputs, mask=None):
        u = torch.tanh(self.W(rnn_outputs))
        scores = self.v(u)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), -1e9)
        alpha = F.softmax(scores, dim=1)
        context = torch.sum(alpha * rnn_outputs, dim=1)
        return context, alpha.squeeze(-1)

class FrozenBERT_BiLSTMAttention(nn.Module):
    def __init__(self, bert_model, hidden_dim=128, num_classes=3):
        super().__init__()
        self.bert = bert_model # Corps BERT gelé
        self.lstm = nn.LSTM(768, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim * 2, attention_dim=64)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.lstm(bert_out)
        mask = (attention_mask == 1)
        context, alpha = self.attention(lstm_out, mask=mask)
        logits = self.fc(self.dropout(context))
        return logits, alpha

# Entraînement de la tête D2
model_d2 = FrozenBERT_BiLSTMAttention(model_ara_frozen, hidden_dim=128).to(device)
optimizer_d2 = torch.optim.Adam(model_d2.parameters(), lr=1e-3)
criterion_d2 = nn.CrossEntropyLoss()

# Dataloader PyTorch pour Transformer
class PairTextDataset(Dataset):
    def __init__(self, targets, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(targets, texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

train_pair_ds = PairTextDataset(train_sub['target'].tolist(), train_sub['text'].tolist(), y_train_id, tokenizer_ara)
val_pair_ds = PairTextDataset(val_df['target'].tolist(), val_df['text'].tolist(), y_val_id, tokenizer_ara)

train_pair_ld = DataLoader(train_pair_ds, batch_size=32, shuffle=True)
val_pair_ld = DataLoader(val_pair_ds, batch_size=64, shuffle=False)

print("Entraînement de la tête BiLSTM+Attention sur AraBERT Gelé (5 époques)...")
for epoch in range(5):
    model_d2.train()
    for batch, y_b in train_pair_ld:
        optimizer_d2.zero_grad()
        logits, _ = model_d2(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = criterion_d2(logits, y_b.to(device))
        loss.backward()
        optimizer_d2.step()

# Évaluation sur Validation
model_d2.eval()
preds_d2 = []
with torch.no_grad():
    for batch, _ in val_pair_ld:
        logits, _ = model_d2(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        preds_d2.extend(torch.argmax(logits, dim=1).cpu().numpy())

m_d2 = compute_stance_metrics(y_val_stance, [id2stance[p] for p in preds_d2])
print("-> D2 : AraBERT Gelé + BiLSTM + Attention :", m_d2)

# Visualisation de l'attention sur 5 exemples
print("--- Visualisation des Poids d'Attention sur 5 Exemples (D2) ---")
for idx in range(5):
    t_text = val_df['text'].iloc[idx]
    t_target = val_df['target'].iloc[idx]
    true_st = val_df['stance'].iloc[idx]

    inp = tokenizer_ara(t_target, t_text, return_tensors="pt").to(device)
    with torch.no_grad():
        _, alpha = model_d2(inp['input_ids'], inp['attention_mask'])

    tokens = tokenizer_ara.convert_ids_to_tokens(inp['input_ids'][0])
    weights = alpha[0].cpu().numpy()

    # Top 4 tokens avec le plus d'attention
    top_indices = np.argsort(weights)[::-1][:4]
    top_tokens = [(tokens[i], round(float(weights[i]), 3)) for i in top_indices]
    print(f"Exemple {idx+1} | Cible: {t_target} | Vraie Stance: {true_st} | Top Tokens d'Attention: {top_tokens}")


Entraînement de la tête BiLSTM+Attention sur AraBERT Gelé (5 époques)...
-> D2 : AraBERT Gelé + BiLSTM + Attention : {'F1_Favor': 0.8409, 'F1_Against': 0.7087, 'F1_None': 0.3929, 'F1_StanceEval': 0.7748, 'Macro_F1': 0.6475}
--- Visualisation des Poids d'Attention sur 5 Exemples (D2) ---
Exemple 1 | Cible: Digital Transformation | Vraie Stance: Favor | Top Tokens d'Attention: [('شغ', 0.258), ('بتم', 0.233), ('"', 0.157), ('##كن', 0.077)]
Exemple 2 | Cible: Digital Transformation | Vraie Stance: Against | Top Tokens d'Attention: [('وتعطيل', 0.366), ('تم', 0.363), ('غلق', 0.116), ('ولماذا', 0.088)]
Exemple 3 | Cible: Covid Vaccine | Vraie Stance: Against | Top Tokens d'Attention: [('لا', 0.678), ('لا', 0.171), ('لتط', 0.054), ('لا', 0.041)]
Exemple 4 | Cible: Women empowerment | Vraie Stance: Against | Top Tokens d'Attention: [('الايجابية', 0.25), ('مرتفعة', 0.199), ('حالها', 0.145), ('بعد', 0.14)]
Exemple 5 | Cible: Digital Transformation | Vraie Stance: None | Top Tokens d'Attention: [(

---
## 5. Partie E1 : Fine-Tuning SOTA : AraBERT vs MARBERTv2 (3 Graines)
Fine-tuning complet de **tous les paramètres** des deux modèles de référence :
* `aubmindlab/bert-base-arabertv02-twitter`
* `UBC-NLP/MARBERTv2`

**Hyperparamètres :**
* Taux d'apprentissage : $2\times 10^{-5}$
* Warmup : 10% des étapes
* Batch size : 32
* Époques : 3
* **3 graines aléatoires (*seeds*) :** 42, 123, 999.


In [5]:
from transformers import AutoModelForSequenceClassification

def finetune_transformer(model_name, seeds=[42, 123, 999], epochs=3, lr=2e-5):
    print(f"\n=======================================================")
    print(f"FINE-TUNING SOTA : {model_name}")
    print(f"=======================================================")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Préparation Dataset avec format en paire
    train_ds = PairTextDataset(train_sub['target'].tolist(), train_sub['text'].tolist(), y_train_id, tokenizer)
    val_ds = PairTextDataset(val_df['target'].tolist(), val_df['text'].tolist(), y_val_id, tokenizer)

    train_ld = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_ld = DataLoader(val_ds, batch_size=64, shuffle=False)

    # Poids de classe pour la perte pondérée
    class_counts = train_sub['stance_id'].value_counts().sort_index().values
    weights = torch.tensor([len(train_sub) / (3.0 * c) for c in class_counts], dtype=torch.float).to(device)
    loss_fct = nn.CrossEntropyLoss(weight=weights)

    seed_scores = []

    for seed in seeds:
        print(f"\n--- Entraînement avec Seed {seed} ---")
        torch.manual_seed(seed)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_ld) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.10), num_training_steps=total_steps)

        for ep in range(epochs):
            model.train()
            for batch, y_b in train_ld:
                optimizer.zero_grad()
                out = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
                loss = loss_fct(out.logits, y_b.to(device))
                loss.backward()
                optimizer.step()
                scheduler.step()

        # Évaluation sur Val
        model.eval()
        preds = []
        with torch.no_grad():
            for batch, _ in val_ld:
                out = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
                preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())

        m = compute_stance_metrics(y_val_stance, [id2stance[p] for p in preds])
        print(f"Seed {seed} -> F1_StanceEval: {m['F1_StanceEval']} | Macro_F1: {m['Macro_F1']}")
        seed_scores.append(m)

    avg_stance = np.mean([s['F1_StanceEval'] for s in seed_scores])
    std_stance = np.std([s['F1_StanceEval'] for s in seed_scores])
    avg_macro = np.mean([s['Macro_F1'] for s in seed_scores])
    print(f"\n==> Résultat Moyen {model_name} : F1_StanceEval = {round(avg_stance, 4)} (+/- {round(std_stance, 4)}) | Macro_F1 = {round(avg_macro, 4)}")
    return seed_scores, avg_stance

# Exécution E1 : AraBERT vs MARBERTv2
scores_arabert, avg_ara = finetune_transformer("aubmindlab/bert-base-arabertv02-twitter", seeds=[42, 123, 999])
scores_marbert, avg_mar = finetune_transformer("UBC-NLP/MARBERTv2", seeds=[42, 123, 999])



FINE-TUNING SOTA : aubmindlab/bert-base-arabertv02-twitter

--- Entraînement avec Seed 42 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Seed 42 -> F1_StanceEval: 0.8113 | Macro_F1: 0.7015

--- Entraînement avec Seed 123 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Seed 123 -> F1_StanceEval: 0.8299 | Macro_F1: 0.7311

--- Entraînement avec Seed 999 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Seed 999 -> F1_StanceEval: 0.8178 | Macro_F1: 0.7278

==> Résultat Moyen UBC-NLP/MARBERTv2 : F1_StanceEval = 0.8225 (+/- 0.0053) | Macro_F1 = 0.7281


---
## 6. Partie E2 : Apprentissage Multi-Tâche (*Multi-Task Learning*)
Entraînement conjoint sur **MARBERTv2** avec 3 têtes :
* **Stance** (Position, 3 classes)
* **Sentiment** (3 classes)
* **Sarcasme** (2 classes)

Nous testons l'effet du coefficient $\lambda$ :
* $\lambda = 0.0$ (Single-task baseline)
* $\lambda = 0.3$ (Équilibre intermédiaire)
* $\lambda = 0.7$ (Forte régularisation auxiliaire)


In [6]:
class MultiTaskDataset(Dataset):
    def __init__(self, targets, texts, stance_ids, sent_ids, sarc_ids, tokenizer, max_len=128):
        self.enc = tokenizer(targets, texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.stance = torch.tensor(stance_ids, dtype=torch.long)
        self.sent = torch.tensor(sent_ids, dtype=torch.long)
        self.sarc = torch.tensor(sarc_ids, dtype=torch.long)
    def __len__(self): return len(self.stance)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        return item, self.stance[i], self.sent[i], self.sarc[i]

class MultiTaskModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        h = self.transformer.config.hidden_size
        self.dropout = nn.Dropout(0.3)
        self.stance_head = nn.Linear(h, 3)
        self.sentiment_head = nn.Linear(h, 3)
        self.sarcasm_head = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = self.dropout(out.last_hidden_state[:, 0, :])
        return self.stance_head(cls_rep), self.sentiment_head(cls_rep), self.sarcasm_head(cls_rep)

tok_mar = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")
mt_train_ds = MultiTaskDataset(train_sub['target'].tolist(), train_sub['text'].tolist(), train_sub['stance_id'].values, train_sub['sentiment_id'].values, train_sub['sarcasm_id'].values, tok_mar)
mt_val_ds = MultiTaskDataset(val_df['target'].tolist(), val_df['text'].tolist(), val_df['stance_id'].values, val_df['sentiment_id'].values, val_df['sarcasm_id'].values, tok_mar)

mt_train_ld = DataLoader(mt_train_ds, batch_size=32, shuffle=True)
mt_val_ld = DataLoader(mt_val_ds, batch_size=64, shuffle=False)

def train_multitask(lam=0.3, epochs=3, lr=2e-5):
    print(f"\n--- Entraînement Multi-Tâche avec lambda = {lam} ---")
    model = MultiTaskModel("UBC-NLP/MARBERTv2").to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    crit_stance = nn.CrossEntropyLoss()
    crit_aux = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()
        for batch, y_st, y_se, y_sa in mt_train_ld:
            optimizer.zero_grad()
            out_st, out_se, out_sa = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = crit_stance(out_st, y_st.to(device)) + lam * crit_aux(out_se, y_se.to(device)) + lam * crit_aux(out_sa, y_sa.to(device))
            loss.backward()
            optimizer.step()

    # Évaluation sur Stance
    model.eval()
    preds = []
    with torch.no_grad():
        for batch, y_st, _, _ in mt_val_ld:
            out_st, _, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(torch.argmax(out_st, dim=1).cpu().numpy())

    m = compute_stance_metrics(y_val_stance, [id2stance[p] for p in preds])
    print(f"Résultats Stance (lambda={lam}) : {m}")
    return m

res_mt_0 = train_multitask(lam=0.0) # Single-task
res_mt_03 = train_multitask(lam=0.3)
res_mt_07 = train_multitask(lam=0.7)



--- Entraînement Multi-Tâche avec lambda = 0.0 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Résultats Stance (lambda=0.0) : {'F1_Favor': 0.8742, 'F1_Against': 0.7771, 'F1_None': 0.4483, 'F1_StanceEval': 0.8256, 'Macro_F1': 0.6999}

--- Entraînement Multi-Tâche avec lambda = 0.3 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Résultats Stance (lambda=0.3) : {'F1_Favor': 0.8884, 'F1_Against': 0.8187, 'F1_None': 0.4615, 'F1_StanceEval': 0.8536, 'Macro_F1': 0.7229}

--- Entraînement Multi-Tâche avec lambda = 0.7 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Résultats Stance (lambda=0.7) : {'F1_Favor': 0.8759, 'F1_Against': 0.7946, 'F1_None': 0.3529, 'F1_StanceEval': 0.8353, 'Macro_F1': 0.6745}


---
## 7. Synthèse & Réponses aux Questions Q8 et Q9 du Rapport

### Réponse à Q8 (PLM Gelé vs Fine-Tuné & AraBERT vs MARBERTv2) :
* **Gelé vs Fine-Tuné (Gain vs Coût) :**
  * Le mode gelé (Partie D) extrait des caractéristiques sans rétropropagation dans BERT. Il est extrêmement rapide et économique, mais plafonne car les couches profondes de BERT ne s'adaptent pas à la spécificité de la Stance.
  * Le fine-tuning complet (Partie E) débloque tous les paramètres et apporte un gain massif de performance (+5 à +10 points sur le F1 StanceEval) au prix d'un temps de calcul plus long.
* **AraBERT-Twitter vs MARBERTv2 :**
  * **MARBERTv2** surpasse généralement AraBERT sur Mawqif.
  * **Pourquoi ?** MARBERTv2 a été pré-entraîné sur 1 milliard de tweets arabes hautement dialectaux. Son vocabulaire et ses représentations encodent naturellement les tournures informelles, le sarcasme et l'argot des réseaux sociaux du Golfe et du Levant, dominants dans ce dataset.

### Réponse à Q9 (Apprentissage Multi-Tâche & Effet de $\lambda$) :
* **L'ajout du sentiment et du sarcasme améliore-t-il la tâche principale ?**
  * Avec un $\lambda$ modéré (ex. $0.3$), la tâche auxiliaire agit comme un **régularisateur** : apprendre à détecter l'ironie empêche le modèle de se faire piéger par des phrases élogieuses au sens second degré.
* **Pourquoi une tâche auxiliaire peut-elle nuire avec un grand $\lambda$ ?**
  * Si $\lambda$ est trop fort (ex. $0.7$ ou plus), il se produit un phénomène de **compétition de gradients (*task interference*)** : le modèle privilégie la classification du sentiment (plus facile et plus explicite) au détriment de la logique plus complexe de la prise de position conditionnée par la cible.
